# 🎙️ PrecisionVoice - Vietnamese Speech-to-Text

Notebook đơn giản để transcribe audio tiếng Việt sử dụng **faster-whisper** và **Gradio UI**.

### Hướng dẫn
1. **Chọn GPU**: `Runtime` → `Change runtime type` → **T4 GPU**
2. **Chạy từng cell** theo thứ tự từ trên xuống
3. **Sử dụng Gradio link** ở cell cuối để truy cập UI

In [1]:
# @title 1. 🔍 Kiểm tra GPU
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU Detected: {gpu_name}")
    print(f"   VRAM: {gpu_mem:.1f} GB")
else:
    print("⚠️ KHÔNG TÌM THẤY GPU!")
    print("👉 Vào Runtime → Change runtime type → T4 GPU")

✅ GPU Detected: Tesla T4
   VRAM: 15.8 GB


In [2]:
# @title 2. 📦 Cài đặt Dependencies
print("Installing dependencies...")
!pip install -q faster-whisper gradio speechbrain scikit-learn librosa
!apt-get install -y -qq ffmpeg > /dev/null 2>&1
print("✅ Dependencies installed successfully!")

Installing dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 21.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.0/38.0 MB 21.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 111.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 8.9 MB/s eta 0:00:00
✅ Dependencies installed successfully!


In [3]:
# @title 3. 🤖 Load Models (Whisper & SpeechBrain)
import torchaudio
import sys

# 1. Monkeypatch torchaudio BEFORE any speechbrain import
if not hasattr(torchaudio, 'list_audio_backends'):
    torchaudio.list_audio_backends = lambda: []

# 2. Handle SpeechBrain initialization quirks
try:
    import speechbrain.utils.quirks
    import speechbrain.utils.profiling
except (ImportError, AttributeError):
    pass

from faster_whisper import WhisperModel
from speechbrain.inference.speaker import EncoderClassifier
import torch
import time

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading EraX-WoW-Turbo model (optimized for Vietnamese)...")
start = time.time()
model = WhisperModel(
    "erax-ai/EraX-WoW-Turbo-V1.1-CT2",
    device=device,
    compute_type="float16" if device == "cuda" else "int8"
)
print(f"✅ Whisper loaded in {time.time() - start:.1f}s")

print("Loading SpeechBrain Speaker Recognition model...")
start = time.time()
classifier = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    run_opts={"device": device}
)
print(f"✅ SpeechBrain loaded in {time.time() - start:.1f}s")

Loading EraX-WoW-Turbo model (optimized for Vietnamese)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

vocabulary.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Model loaded in 28.5s


In [ ]:
# @title 4. 🎤 Khởi chạy Gradio UI
import gradio as gr
import time
import torch
import numpy as np
import librosa
from sklearn.cluster import SpectralClustering

def diarize_segments(audio_path, segments, num_speakers):
    """Assign speaker labels to Whisper segments using SpeechBrain embeddings."""
    if not segments:
        return []
    
    # Load audio
    audio, sr = librosa.load(audio_path, sr=16000)
    
    embeddings = []
    valid_segments = []
    
    for segment in segments:
        # Extract segment audio
        start_sample = int(segment.start * sr)
        end_sample = int(segment.end * sr)
        
        if end_sample - start_sample < 160: # 10ms minimum
            continue
            
        seg_audio = audio[start_sample:end_sample]
        
        # Get embedding
        with torch.no_grad():
            seg_tensor = torch.tensor(seg_audio).unsqueeze(0)
            emb = classifier.encode_batch(seg_tensor)
            embeddings.append(emb.squeeze().cpu().numpy())
            valid_segments.append(segment)
    
    if not embeddings:
        return segments
        
    # Clustering
    clustering = SpectralClustering(
        n_clusters=num_speakers,
        affinity='cosine',
        random_state=42
    )
    labels = clustering.fit_predict(embeddings)
    
    # Map labels back
    result = []
    for i, segment in enumerate(valid_segments):
        segment_dict = {
            'start': segment.start,
            'end': segment.end,
            'text': segment.text,
            'speaker': f"Speaker {labels[i]}"
        }
        result.append(segment_dict)
        
    return result

def transcribe(audio_path, language, beam_size, vad_filter, num_speakers):
    """Transcribe audio file to text with speaker diarization."""
    if audio_path is None:
        return "⚠️ Vui lòng upload hoặc ghi âm audio!"
    
    start_time = time.time()
    
    # 1. Transcribe with Whisper
    segments_gen, info = model.transcribe(
        audio_path,
        language=language if language != "auto" else None,
        beam_size=beam_size,
        vad_filter=vad_filter,
        word_timestamps=False,
    )
    segments = list(segments_gen)
    
    # 2. Diarize with SpeechBrain
    diarized_segments = diarize_segments(audio_path, segments, num_speakers)
    
    # 3. Format output
    result_lines = []
    full_text = []
    
    for seg in diarized_segments:
        speaker = seg.get('speaker', 'Unknown')
        result_lines.append(f"[{seg['start']:.2f}s → {seg['end']:.2f}s] **{speaker}**: {seg['text']}")
        full_text.append(f"{speaker}: {seg['text'].strip()}")
    
    elapsed = time.time() - start_time
    
    # Summary
    output = f"📊 Detected language: {info.language} (prob: {info.language_probability:.2%})\n"
    output += f"⏱️ Processing time: {elapsed:.1f}s\n"
    output += f"━" * 50 + "\n\n"
    output += "\n".join(result_lines)
    output += f"\n\n" + "━" * 50 + "\n"
    output += f"📝 Full transcript:\n{'\n'.join(full_text)}"
    
    return output

# Build UI
with gr.Blocks(title="PrecisionVoice", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎙️ PrecisionVoice - Vietnamese STT (Diarization supported)")
    gr.Markdown("Upload audio hoặc ghi âm trực tiếp từ microphone để xem ai đang nói gì.")
    
    with gr.Row():
        with gr.Column(scale=1):
            audio_input = gr.Audio(
                sources=["upload", "microphone"],
                type="filepath",
                label="🔊 Audio Input"
            )
            
            with gr.Row():
                language = gr.Dropdown(
                    choices=["auto", "vi", "en", "zh", "ja", "ko"],
                    value="vi",
                    label="🌐 Language"
                )
                num_speakers = gr.Slider(
                    minimum=1, maximum=10, value=2, step=1,
                    label="👥 Number of Speakers"
                )
            
            with gr.Accordion("Advanced Settings", open=False):
                beam_size = gr.Slider(
                    minimum=1, maximum=10, value=5, step=1,
                    label="🎯 Beam Size"
                )
                vad_filter = gr.Checkbox(
                    value=True,
                    label="🔇 VAD Filter (lọc khoảng lặng)"
                )
            
            transcribe_btn = gr.Button("▶️ Transcribe & Diarize", variant="primary")
        
        with gr.Column(scale=2):
            output_text = gr.Markdown(
                label="📝 Transcription & Diarization Result"
            )
    
    transcribe_btn.click(
        fn=transcribe,
        inputs=[audio_input, language, beam_size, vad_filter, num_speakers],
        outputs=output_text
    )
    
    gr.Markdown("---")
    gr.Markdown("*Model: EraX-WoW-Turbo & SpeechBrain ECAPA-TDNN | Powered by faster-whisper*")
try:
    shell = get_ipython().__class__.__name__
    if shell == 'ZMQInteractiveShell':
        launch_args = {"share": True, "debug": True}
    else:
        launch_args = {"share": False}
except NameError:
    launch_args = {"share": False}
demo.launch(**launch_args)

/tmp/ipython-input-130105980.py:42: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="PrecisionVoice", theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://76b49811a61ce08b19.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
